# Testing a Translation Error Checker

This notebook is a runnable demo of `method.py`, which validates a **four-category content-invariant checker** for machine-translation errors against the `checker_validation_heldout` fold of a WMT25 translation error-correction dataset, across six target languages: `zh_CN`, `cs_CZ`, `ja_JP`, `is_IS`, `ru_RU`, `uk_UA`.

**The four categories:**
- `named_entity_swap` — a named entity in the translation is swapped for another
- `number_unit_date_alteration` — a number, date, unit, or currency amount is altered
- `negation_polarity_flip` — a negation cue is deleted, flipping the meaning
- `quantifier_substitution` — a quantifier word (e.g. "all" → "none") is swapped for its antonym

**Two systems are run side by side** on the identical held-out rows:
- **OUR METHOD**: `checker_module.py`'s resourced checker — stanza NER for named-entity detection where a trained model exists (zh/ja/ru/uk), a capitalization-heuristic regex fallback for cs/is (stanza ships no NER model for either), a uniform regex parser for numbers/units/dates/currency, a cross-lingual negation-polarity mismatch check, and closed-class quantifier word-list matching.
- **BASELINE**: a language-naive generic checker — a Latin-only capitalization regex applied uniformly regardless of script, a bare-digit number regex, and *no* negation/quantifier resource at all — representing an off-the-shelf tool with zero per-language investment.

For every injected row (which carries a ground-truth corruption span) we check whether each system's flagged spans overlap the ground truth (**recall**). For natural (uncorrupted) rows we count each system's flags that fall outside CometKiwi's QE-flagged spans as a noisy false-positive proxy (**precision**). Both are reported per `(language, category)` cell with bootstrap 95% confidence intervals, exactly as in `method.py`.

**Demo-scale note:** the full run downloads stanza NER models for zh/ja/ru/uk (~100s of MB each) and evaluates 448 injected + 1080 natural×4-category rows (~7 minutes). To keep this notebook fast and Colab-friendly, we deliberately do **not** install `stanza`/`torch` here — `checker_module.py`'s own `except ImportError` branch (unchanged) then gracefully falls back to the regex/capitalization heuristic for **all** languages (not just cs/is), and we run over a small curated subset of the held-out rows. The code path exercised is identical to the full run; only the entity-detector resourcing and data volume are reduced.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# loguru, psutil -- NOT pre-installed on Colab, always install
_pip('loguru==0.7.3')
_pip('psutil==7.2.2')

# numpy, matplotlib -- pre-installed on Colab, install locally only (Colab's exact versions)
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'matplotlib==3.10.0')

# NOTE: we deliberately do NOT install stanza/torch here (see markdown above) -- this
# keeps the notebook fast, and checker_module.py's own `except ImportError` fallback
# handles the "stanza unavailable" case exactly as designed, with no code changes.

## `checker_module.py`

This is the artifact's importable checker module, written to disk here **verbatim** (unchanged) so the notebook is self-contained on Colab, then imported exactly as `method.py` does (`import checker_module as cm`).

In [ ]:
%%writefile checker_module.py
#!/usr/bin/env python3
"""checker_module.py -- importable, deterministic content-invariant checker.

Implements the four-category checker (named_entity_swap, number_unit_date_alteration,
negation_polarity_flip, quantifier_substitution) across {zh_CN, cs_CZ, ja_JP, is_IS,
ru_RU, uk_UA}, plus a language-naive BASELINE checker for comparison.

Every detector is a pure function ``detect(source_text, target_text, lang) -> list[(start, end)]``
returning character-offset spans INTO `target_text` that the checker flags as candidate
sites of that invariant category. Detectors are independent of any specific dataset row;
they only see the (English source, target-language output) pair, exactly as a checker
would at inference/QE time -- they never see the ground-truth corruption metadata.

Resourcing decisions (see the resource dossier, art_5ySTX4YfxqG_):
  - named_entity_swap: stanza NER for languages stanza covers with a trained NER model
    (zh, ja, ru, uk per the dossier's correction of Stanza's coverage); a documented
    regex/capitalization fallback for languages stanza does NOT ship NER for (cs, is --
    exactly the gap the dossier flags: NameTag3/IceBERT would close it but require a
    non-Python API dependency / gated download respectively, so per FALLBACK_PLAN this
    substitutes a heuristic and logs it rather than silently degrading).
  - number_unit_date_alteration: regex only, uniformly across all six languages, per
    FALLBACK_PLAN ("if Duckling is not installable/reachable ... fall back uniformly to
    the regex-based number/unit/date parser for all six languages"). Digits, dates and
    currency symbols are script-independent, so one regex set serves every language.
  - negation_polarity_flip: corruption is a DELETION (the single negation cue in the
    segment is removed -- construction guarantees exactly one cue was present, so
    target-only presence detection at the ground-truth offset is structurally impossible:
    there is nothing left there to point at). Implemented instead as a CROSS-LINGUAL
    polarity-mismatch check: English negation-cue presence in `source` vs. target-language
    negation-cue presence in `target`; a mismatch flags the whole segment. This is the
    SENTENCE-LEVEL fallback the artifact plan's own fallback_plan anticipates.
  - quantifier_substitution: corruption is an IN-PLACE word swap (kept at the same start
    offset), so target-only span detection works directly: flag every occurrence of any
    known closed-class quantifier word.
"""

from __future__ import annotations

import re
from dataclasses import dataclass, field

Span = tuple[int, int]

LANGUAGES = ["zh_CN", "cs_CZ", "ja_JP", "is_IS", "ru_RU", "uk_UA"]

WORD_BOUNDARY_LANGS = {"cs_CZ", "is_IS", "ru_RU", "uk_UA"}  # space-delimited scripts

# ---------------------------------------------------------------------------
# Category 2: number / unit / date alteration -- regex, script-independent
# ---------------------------------------------------------------------------

_YEAR_RE = re.compile(r"(?<![@\w])(1[5-9]\d{2}|20\d{2}|21\d{2})(?!\w)")
_DATE_RE = re.compile(r"(?<!\w)\d{1,4}[./-]\d{1,2}[./-]\d{1,4}(?!\w)")
_CURRENCY_RE = re.compile(
    r"[$€£¥₽₴]\s?\d[\d,.\s]*"
    r"|\d[\d,.\s]*\s?(?:USD|EUR|CZK|ISK|RUB|UAH|JPY|CNY|Kč|kr)\b",
    re.IGNORECASE,
)
_NUMBER_RE = re.compile(r"(?<![@\w.,])\d+(?:[.,]\d+)?%?(?!\w)")


def _dedup_overlaps(spans: set[Span]) -> list[Span]:
    """Keep the widest span at each overlapping cluster (e.g. a date subsumes a bare year)."""
    ordered = sorted(spans, key=lambda s: (s[0], -(s[1] - s[0])))
    out: list[Span] = []
    for s in ordered:
        if any(s[0] >= o[0] and s[1] <= o[1] for o in out):
            continue
        out.append(s)
    return sorted(out)


def detect_number_unit_date(source: str, target: str, lang: str) -> list[Span]:
    spans: set[Span] = set()
    for rx in (_DATE_RE, _CURRENCY_RE, _YEAR_RE, _NUMBER_RE):
        for m in rx.finditer(target):
            if m.end() > m.start():
                spans.add((m.start(), m.end()))
    return _dedup_overlaps(spans)


def detect_number_unit_date_baseline(source: str, target: str, lang: str) -> list[Span]:
    """Language-naive baseline: bare digit runs only, no date/currency/year awareness."""
    return _dedup_overlaps({(m.start(), m.end()) for m in re.finditer(r"\d+(?:[.,]\d+)?", target)})


# ---------------------------------------------------------------------------
# Category 1: named entity swap
# ---------------------------------------------------------------------------

CS_UPPER, CS_LOWER = "ÁČĎÉĚÍŇÓŘŠŤÚŮÝŽ", "áčďéěíňóřšťúůýž"
IS_UPPER, IS_LOWER = "ÁÐÉÍÓÚÝÞÆÖ", "áðéíóúýþæö"
_LATIN_ENTITY_RE = re.compile(
    rf"[A-Z{CS_UPPER}{IS_UPPER}][a-z{CS_LOWER}{IS_LOWER}]+(?:\s[A-Z{CS_UPPER}{IS_UPPER}][a-z{CS_LOWER}{IS_LOWER}]+){{0,2}}"
)
_CYRILLIC_ENTITY_RE = re.compile(r"[А-ЯЁІЇЄҐ][а-яёіїєґ'-]+(?:\s[А-ЯЁІЇЄҐ][а-яёіїєґ'-]+){0,2}")

# Independently-curated closed-class stopwords (demonstratives / pronouns / conjunctions /
# frequent sentence-initial adverbs) that a naive capitalization heuristic must exclude for
# cs_CZ / is_IS -- these are the two languages stanza has no trained NER model for.
_ENTITY_STOPWORDS = {
    "cs_CZ": {"Tato", "Tento", "Tyto", "Toto", "Ten", "Ta", "To", "Ale", "Avšak", "Proto",
              "Podle", "Když", "Pokud", "Nicméně", "Také", "Proč", "Jak", "Kdy", "Kde", "Co",
              "Kdo", "My", "Vy", "Oni", "Ona", "On", "Jeho", "Její", "Jejich", "Náš", "Váš",
              "Ještě", "Už", "Teď", "Poté", "Potom", "Přesto", "Protože", "Takže"},
    "is_IS": {"Þetta", "Þessi", "Þessar", "Þessu", "Sá", "Sú", "Það", "Þau", "En", "Því",
              "Samt", "Samkvæmt", "Ef", "Þegar", "Hvers", "Hvernig", "Hvenær", "Hvar", "Hvað",
              "Hver", "Við", "Þið", "Þeir", "Hún", "Hann", "Hans", "Hennar", "Þeirra", "Okkar",
              "Enn", "Núna", "Síðan"},
}
_STOP_LATIN_GENERIC = {"The", "This", "That", "These", "Those", "A", "An", "It", "He", "She",
                        "They", "We", "You", "I"}


def _is_sentence_start(text: str, start: int) -> bool:
    prefix = text[:start].rstrip()
    return not prefix or prefix[-1] in ".!?…”’\""


def detect_entity_regex(source: str, target: str, lang: str) -> list[Span]:
    """Capitalization-heuristic entity detector used where no trained NER model is available."""
    if lang in ("ru_RU", "uk_UA"):
        rx = _CYRILLIC_ENTITY_RE
        stop: set[str] = set()
    else:
        rx = _LATIN_ENTITY_RE
        stop = _ENTITY_STOPWORDS.get(lang, set()) | _STOP_LATIN_GENERIC
    out = []
    for m in rx.finditer(target):
        if len(m.group()) < 3 or m.group() in stop:
            continue
        if _is_sentence_start(target, m.start()):
            continue
        out.append((m.start(), m.end()))
    return out


def detect_entity_baseline(source: str, target: str, lang: str) -> list[Span]:
    """Language-naive baseline: Latin-capitalization regex applied uniformly to every
    language regardless of script, with no stopword filtering -- a generic off-the-shelf
    heuristic with zero localization effort."""
    return [(m.start(), m.end()) for m in _LATIN_ENTITY_RE.finditer(target) if len(m.group()) >= 3]


# ---------------------------------------------------------------------------
# Category 3: negation polarity flip -- cross-lingual sentence-level check
# ---------------------------------------------------------------------------

_EN_NEG_RE = re.compile(
    r"\b(?:not|never|none|nobody|nothing|neither|nor|without|hardly|barely|scarcely|"
    r"cannot|no)\b|n't",
    re.IGNORECASE,
)

NEGATION_CUES = {
    "cs_CZ": ["nikdy", "žádný", "žádná", "žádné", "žádného", "žádnou", "nic", "nikoho",
              "není", "nejsou", "nemá", "nemají", "nebude", "nechce", "neví", "nemůže",
              "neudělal", "nedělá"],
    "is_IS": ["ekki", "aldrei", "enginn", "engin", "ekkert", "hvorki", "engum", "engan"],
    "ja_JP": ["ない", "ません", "なかった",
              "ありません", "できない"],
    "ru_RU": ["не", "нет", "никогда", "ничего", "никто", "нельзя"],
    "uk_UA": ["не", "ні", "ніколи", "нічого", "ніхто"],
    "zh_CN": ["不", "没有", "没", "从不", "无法", "并非", "未"],
}


def _cue_occurrences(text: str, cue: str, lang: str) -> list[re.Match]:
    if lang in WORD_BOUNDARY_LANGS:
        return list(re.finditer(rf"\b{re.escape(cue)}\b", text))
    return list(re.finditer(re.escape(cue), text))


def detect_negation_flip(source: str, target: str, lang: str) -> list[Span]:
    src_hits = len(list(_EN_NEG_RE.finditer(source)))
    cues = NEGATION_CUES.get(lang, [])
    tgt_hits = sum(len(_cue_occurrences(target, c, lang)) for c in cues)
    src_has_neg = src_hits >= 1
    tgt_has_neg = tgt_hits >= 1
    if src_has_neg != tgt_has_neg:
        return [(0, len(target))]  # sentence-level flag: polarity mismatch, per fallback_plan
    return []


def detect_negation_baseline(source: str, target: str, lang: str) -> list[Span]:
    """Language-naive baseline: no per-language negation-cue resource -> never fires."""
    return []


# ---------------------------------------------------------------------------
# Category 4: quantifier substitution -- target-only closed-class word list
# ---------------------------------------------------------------------------

QUANTIFIER_WORDS = {
    "cs_CZ": ["všichni", "všechny", "všechna", "někteří", "některé", "některá", "vždy",
              "nikdy", "nikdo", "někdo", "nic", "něco", "občas", "každý", "žádný"],
    "is_IS": ["allir", "allar", "öll", "sumir", "sumar", "alltaf", "aldrei", "enginn",
              "einhver", "ekkert", "eitthvað", "hver"],
    "ja_JP": ["すべて", "すべての", "一部の",
              "いつも", "決して", "誰も", "誰か",
              "常に", "時々"],
    "ru_RU": ["все", "некоторые", "всегда", "никогда", "никто", "кто-то", "ничего",
              "что-то", "каждый", "любой"],
    "uk_UA": ["усі", "всі", "деякі", "завжди", "ніколи", "ніхто", "хтось", "кожен",
              "будь-який"],
    "zh_CN": ["所有", "一些", "总是", "从不", "每个",
              "某些", "从来没有", "有时", "任何"],
}


def detect_quantifier(source: str, target: str, lang: str) -> list[Span]:
    words = QUANTIFIER_WORDS.get(lang, [])
    out = []
    for w in words:
        for m in _cue_occurrences(target, w, lang):
            out.append((m.start(), m.end()))
    return _dedup_overlaps(set(out))


def detect_quantifier_baseline(source: str, target: str, lang: str) -> list[Span]:
    """Language-naive baseline: no per-language quantifier-word resource -> never fires."""
    return []


# ---------------------------------------------------------------------------
# Registry
# ---------------------------------------------------------------------------

CATEGORY_KEYS = [
    "named_entity_swap",
    "number_unit_date_alteration",
    "negation_polarity_flip",
    "quantifier_substitution",
]

BASELINE_DETECTORS = {
    "named_entity_swap": detect_entity_baseline,
    "number_unit_date_alteration": detect_number_unit_date_baseline,
    "negation_polarity_flip": detect_negation_baseline,
    "quantifier_substitution": detect_quantifier_baseline,
}


@dataclass
class CheckerRegistry:
    """Holds the resolved entity detector per language (stanza NER where available,
    regex fallback otherwise) plus the three regex/rule detectors shared across languages."""

    entity_fn_by_lang: dict = field(default_factory=dict)
    substitutions: dict = field(default_factory=dict)  # lang -> reason string, logged

    def detect(self, category: str, source: str, target: str, lang: str) -> list[Span]:
        if category == "named_entity_swap":
            fn = self.entity_fn_by_lang.get(lang, detect_entity_regex)
            return fn(source, target, lang)
        if category == "number_unit_date_alteration":
            return detect_number_unit_date(source, target, lang)
        if category == "negation_polarity_flip":
            return detect_negation_flip(source, target, lang)
        if category == "quantifier_substitution":
            return detect_quantifier(source, target, lang)
        raise ValueError(f"unknown category {category}")

    def detect_baseline(self, category: str, source: str, target: str, lang: str) -> list[Span]:
        return BASELINE_DETECTORS[category](source, target, lang)


def overlaps(a: Span, b: Span) -> bool:
    return a[0] < b[1] and b[0] < a[1]


def any_overlap(spans: list[Span], gt: Span) -> bool:
    return any(overlaps(s, gt) for s in spans)

## Imports

Copied from `method.py`'s import block, plus `matplotlib` for the results visualization at the end.

In [ ]:
import gc
import sys
import time
from collections import defaultdict
from pathlib import Path

import numpy as np
import psutil
from loguru import logger
import matplotlib.pyplot as plt

import checker_module as cm

logger.remove()
logger.add(sys.stdout, level="INFO", format="{time:HH:mm:ss}|{level:<7}|{message}")

RNG = np.random.default_rng(20260901)

LANG_CODE = {  # metadata_language_pair "en-xx_XX" -> checker_module language key
    "en-zh_CN": "zh_CN", "en-cs_CZ": "cs_CZ", "en-ja_JP": "ja_JP",
    "en-is_IS": "is_IS", "en-ru_RU": "ru_RU", "en-uk_UA": "uk_UA",
}

## Data loading

`mini_demo_data.json` is a curated subset of the `checker_validation_heldout` fold (48 injected rows -- up to 2 per (language, category) cell -- plus 36 natural rows, 6 per language), kept in the exact same raw structure `method.py`'s `load_rows()` expects. We try the GitHub-hosted copy first (for Colab), falling back to the local file.

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/ai-inventor-outputs/ai-invention-e9bf19-isolating-what-fixes-failed-span-editing/main/round-2/experiment-1/demo/mini_demo_data.json"
import json, os

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception: pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f: return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
raw_data = load_data()
print("datasets:", [ds["dataset"] for ds in raw_data["datasets"]])
print("total examples:", sum(len(ds["examples"]) for ds in raw_data["datasets"]))

## Config

All tunable parameters from `method.py`, gathered in one place. `N_BOOT` (bootstrap resamples) is the one true compute knob -- it is cheap even at the full 2000 since it is a vectorized numpy operation over tiny per-cell arrays, so we keep it at its original value. `LIMIT_INJECTED`/`LIMIT_NATURAL` cap how many rows to process (here: `None` = use the whole curated mini dataset, which is already small). `RAM_GB` and `USE_GPU` mirror the original CLI defaults; `USE_GPU=False` matches the original (`args.gpu` defaults to `False`).

To scale this notebook up towards the full run: point `load_data()`'s local fallback at `full_method_out.json`'s upstream input (`full_data_out.json`, 448 injected + 1080 natural rows) and set `LIMIT_INJECTED`/`LIMIT_NATURAL` to `None`, then also install `stanza`+`torch` in the install cell to get the real NER coverage back.

In [ ]:
# --- config (tunable) ---
N_BOOT = 2000              # bootstrap resamples for CIs (original value; cheap regardless of n)
ALPHA = 0.05                # 95% CI
USABLE_THRESHOLD = 0.5      # a cell is excluded from headline if precision or recall falls below this
LIMIT_INJECTED = None        # cap on injected rows processed (None = use full curated mini set)
LIMIT_NATURAL = None         # cap on natural rows processed (None = use full curated mini set)
RAM_GB = 4.0                 # RAM budget passed to set_resource_limits (original default: 20.0)
USE_GPU = False               # matches original --no-gpu default

## Resource limits

Caps the process's address space so an unexpectedly large run fails fast with `MemoryError` instead of getting OOM-killed. Copied unchanged from `method.py`.

In [ ]:
import resource


def set_resource_limits(ram_gb: float) -> None:
    avail = psutil.virtual_memory().available
    budget = int(ram_gb * 1e9)
    if budget >= avail:
        budget = int(avail * 0.7)
    resource.setrlimit(resource.RLIMIT_AS, (budget * 3, budget * 3))
    logger.info(f"RAM budget set to {budget / 1e9:.1f} GB (available {avail / 1e9:.1f} GB)")


set_resource_limits(ram_gb=RAM_GB)

## Entity-detector registry (stanza NER + regex fallback)

Builds the `CheckerRegistry` that resolves, per language, which named-entity detector `detect(...)` uses: a stanza NER pipeline where one is loaded, or `checker_module.detect_entity_regex` otherwise. Copied unchanged from `method.py`'s `build_entity_registry` -- since we did not install `stanza` above, the `except ImportError` branch fires and every language logs a substitution reason (exactly the graceful degradation the original code was written to handle).

In [ ]:
STANZA_NER_LANGS = {"zh_CN": "zh", "ja_JP": "ja", "ru_RU": "ru", "uk_UA": "uk"}


def build_entity_registry(use_gpu: bool) -> cm.CheckerRegistry:
    reg = cm.CheckerRegistry()
    try:
        import stanza
    except ImportError:
        logger.warning("stanza not importable -- ALL languages fall back to regex entity heuristic")
        for lang in cm.LANGUAGES:
            reg.substitutions[lang] = "stanza unavailable (import failed) -> regex capitalization heuristic"
        return reg

    for lang_key, stanza_code in STANZA_NER_LANGS.items():
        try:
            t0 = time.time()
            stanza.download(stanza_code, processors="tokenize,ner", verbose=False)
            nlp = stanza.Pipeline(
                stanza_code, processors="tokenize,ner", use_gpu=use_gpu, verbose=False,
                tokenize_no_ssplit=False,
            )
            logger.info(f"stanza NER pipeline for {lang_key} ({stanza_code}) ready in {time.time() - t0:.1f}s")

            def make_fn(nlp_pipeline, code=lang_key):
                def fn(source: str, target: str, lang: str) -> list[tuple[int, int]]:
                    doc = nlp_pipeline(target)
                    spans = []
                    for ent in doc.ents:
                        spans.append((ent.start_char, ent.end_char))
                    return spans
                return fn

            reg.entity_fn_by_lang[lang_key] = make_fn(nlp)
        except Exception as e:  # noqa: BLE001 -- any model/network failure must not kill the run
            logger.error(f"stanza NER load FAILED for {lang_key} ({stanza_code}): {e}")
            reg.substitutions[lang_key] = f"stanza NER load failed ({e}) -> regex capitalization heuristic"

    for lang in ("cs_CZ", "is_IS"):
        reg.substitutions[lang] = (
            "stanza ships no trained NER model for this language (dossier-confirmed gap; "
            "NameTag3/IceBERT would close it but need a non-Python API / gated download) "
            "-> regex capitalization heuristic per fallback_plan"
        )
    return reg


logger.info("=== building entity-detector registry (stanza NER + regex fallback) ===")
reg = build_entity_registry(use_gpu=USE_GPU)
for lang, reason in reg.substitutions.items():
    logger.warning(f"SUBSTITUTION[{lang}] entity detector: {reason}")

## Splitting rows into natural vs. injected

Adapted from `method.py`'s `load_rows` -- the only change is that it takes the already-loaded `raw_data` dict instead of reading a file path (the file path is what the GitHub/local `load_data()` helper above replaces).

In [ ]:
def load_rows(raw: dict, limit_injected: int | None = None) -> tuple[list[dict], list[dict]]:
    natural, injected = [], []
    for ds in raw["datasets"]:
        for ex in ds["examples"]:
            if ex.get("metadata_fold") != "checker_validation_heldout":
                continue
            if ex["metadata_provenance"] == "natural":
                natural.append(ex)
            else:
                injected.append(ex)
    if limit_injected is not None:
        injected = injected[:limit_injected]
    logger.info(f"heldout rows: natural={len(natural)} injected={len(injected)}")
    return natural, injected


t_start = time.time()
natural, injected = load_rows(raw_data, limit_injected=LIMIT_INJECTED)
if LIMIT_NATURAL is not None:
    natural = natural[:LIMIT_NATURAL]

## Scoring: OUR method vs. BASELINE on injected and natural rows

`score_injected_rows` runs both checkers on every injected row and checks span overlap against the ground-truth corruption span (the direct **recall** signal). `score_natural_rows` runs both checkers on every natural row, for every category, and counts flagged spans that fall outside any CometKiwi QE-flagged span as the noisy **precision** false-positive proxy. Copied unchanged from `method.py`.

In [ ]:
def score_injected_rows(injected: list[dict], reg: cm.CheckerRegistry) -> list[dict]:
    """Run both checkers over every injected heldout row; return per-row results."""
    out = []
    for ex in injected:
        lp = ex["metadata_language_pair"]
        lang = LANG_CODE.get(lp)
        cat = ex["metadata_invariant_category"]
        if lang is None or cat not in cm.CATEGORY_KEYS:
            continue
        source = ex["input"]
        target = ex["output"]
        gt_off = ex["metadata_span_offsets"]
        gt_span = (gt_off["start"], min(gt_off["end"], len(target)))

        our_spans = reg.detect(cat, source, target, lang)
        base_spans = reg.detect_baseline(cat, source, target, lang)

        our_hit = cm.any_overlap(our_spans, gt_span)
        base_hit = cm.any_overlap(base_spans, gt_span)

        out.append({
            "language_pair": lp, "lang": lang, "category": cat,
            "gt_span": gt_span, "our_spans": our_spans, "base_spans": base_spans,
            "our_hit": our_hit, "base_hit": base_hit,
            "our_extra_flags": max(0, len(our_spans) - (1 if our_hit else 0)),
            "base_extra_flags": max(0, len(base_spans) - (1 if base_hit else 0)),
            "example": ex,
        })
    return out


def score_natural_rows(natural: list[dict], reg: cm.CheckerRegistry) -> list[dict]:
    """Secondary/noisier FP check: run both checkers over every natural heldout row for
    every category, count flags that fall OUTSIDE any CometKiwi qe_flagged_span."""
    out = []
    for ex in natural:
        lp = ex["metadata_language_pair"]
        lang = LANG_CODE.get(lp)
        if lang is None:
            continue
        source, target = ex["input"], ex["output"]
        qe_spans = ex.get("metadata_qe_flagged_spans") or []
        qe_spans = [(s["start_i"], s["end_i"]) for s in qe_spans]
        for cat in cm.CATEGORY_KEYS:
            our_spans = reg.detect(cat, source, target, lang)
            base_spans = reg.detect_baseline(cat, source, target, lang)
            our_fp = sum(1 for s in our_spans if not any(cm.overlaps(s, q) for q in qe_spans))
            base_fp = sum(1 for s in base_spans if not any(cm.overlaps(s, q) for q in qe_spans))
            out.append({
                "language_pair": lp, "lang": lang, "category": cat,
                "our_n_flags": len(our_spans), "base_n_flags": len(base_spans),
                "our_fp": our_fp, "base_fp": base_fp,
            })
    return out


logger.info(f"=== scoring {len(injected)} injected heldout rows (primary signal) ===")
t0 = time.time()
scored_injected = score_injected_rows(injected, reg)
logger.info(f"scored injected rows in {time.time() - t0:.1f}s")

logger.info(f"=== scoring {len(natural)} natural heldout rows x 4 categories (secondary FP check) ===")
t0 = time.time()
natural_flags = score_natural_rows(natural, reg)
logger.info(f"scored natural rows in {time.time() - t0:.1f}s")

## Bootstrap CIs and the per-cell (language, category) table

`bootstrap_ci` resamples the hit/miss vector `N_BOOT` times to get a 95% CI on recall. `build_table` assembles one row per `(language, category)` cell with `n_injected`, `precision`, `recall`, both CIs, and an `excluded_from_headline` flag when either metric is below `USABLE_THRESHOLD`. Copied unchanged from `method.py` (module-level `N_BOOT`/`ALPHA`/`USABLE_THRESHOLD` reads now come from the config cell above).

In [ ]:
def bootstrap_ci(hits: np.ndarray, n_boot: int, alpha: float) -> tuple[float, float]:
    n = len(hits)
    if n == 0:
        return (float("nan"), float("nan"))
    idx = RNG.integers(0, n, size=(n_boot, n))
    boot_means = hits[idx].mean(axis=1)
    lo = float(np.percentile(boot_means, 100 * alpha / 2))
    hi = float(np.percentile(boot_means, 100 * (1 - alpha / 2)))
    return lo, hi


def build_table(scored_injected: list[dict], natural_flags: list[dict], system: str) -> list[dict]:
    hit_key = f"{system}_hit"
    fp_key = f"{system}_fp"
    n_flags_key = f"{system}_n_flags"

    by_cell = defaultdict(list)
    for r in scored_injected:
        by_cell[(r["lang"], r["category"])].append(r)

    nat_by_cell = defaultdict(lambda: {"fp": 0, "flags": 0, "n_rows": 0, "fp_per_row": []})
    for r in natural_flags:
        c = nat_by_cell[(r["lang"], r["category"])]
        c["fp"] += r[fp_key]
        c["flags"] += r[n_flags_key]
        c["n_rows"] += 1
        c["fp_per_row"].append(r[fp_key])

    table = []
    for lang in cm.LANGUAGES:
        for cat in cm.CATEGORY_KEYS:
            cell_rows = by_cell.get((lang, cat), [])
            n = len(cell_rows)
            if n == 0:
                table.append({
                    "language": lang, "category": cat, "system": system,
                    "n_injected": 0, "precision": None, "recall": None,
                    "ci_precision": [None, None], "ci_recall": [None, None],
                    "excluded_from_headline": True,
                    "exclusion_reason": "no checker_validation_heldout rows for this (language, category) cell",
                })
                continue
            hits = np.array([1.0 if r[hit_key] else 0.0 for r in cell_rows])
            tp = int(hits.sum())
            recall = tp / n
            fp = nat_by_cell[(lang, cat)]["fp"]
            precision = tp / (tp + fp) if (tp + fp) > 0 else (1.0 if fp == 0 else 0.0)
            ci_r = bootstrap_ci(hits, N_BOOT, ALPHA)
            # precision CI: resample injected TP rows AND natural FP-count-per-row rows
            # independently (each at their own n), then recombine -- a row-level
            # bootstrap, not a binomial approximation (fp is a per-row SPAN COUNT that
            # can exceed 1, so treating fp/n_nat as a Bernoulli rate is invalid).
            n_nat = nat_by_cell[(lang, cat)]["n_rows"]
            fp_per_row = np.array(nat_by_cell[(lang, cat)]["fp_per_row"], dtype=float)
            if n_nat > 0 and (tp + fp) > 0:
                boot_tp = RNG.integers(0, n, size=(N_BOOT, n))
                boot_tp_counts = hits[boot_tp].sum(axis=1)
                boot_nat = RNG.integers(0, n_nat, size=(N_BOOT, n_nat))
                boot_fp_counts = fp_per_row[boot_nat].sum(axis=1)
                denom = boot_tp_counts + boot_fp_counts
                with np.errstate(invalid="ignore", divide="ignore"):
                    boot_prec = np.where(denom > 0, boot_tp_counts / np.maximum(denom, 1), np.nan)
                valid = boot_prec[~np.isnan(boot_prec)]
                ci_p = (
                    (float(np.percentile(valid, 100 * ALPHA / 2)), float(np.percentile(valid, 100 * (1 - ALPHA / 2))))
                    if len(valid) > 10 else (float("nan"), float("nan"))
                )
            else:
                ci_p = (float("nan"), float("nan"))

            n_boot_note = None
            if n < 10:
                n_boot_note = f"n={n} < 10: point estimate reported, CI is WIDE/UNSTABLE, not to be over-interpreted"

            excluded = (recall < USABLE_THRESHOLD) or (precision is not None and precision < USABLE_THRESHOLD)
            reason = None
            if excluded:
                reason = f"recall={recall:.3f} or precision={precision:.3f} below usability threshold {USABLE_THRESHOLD}"

            table.append({
                "language": lang, "category": cat, "system": system,
                "n_injected": n, "n_natural_checked": n_nat,
                "true_positives": tp, "false_positives_natural": fp,
                "precision": round(precision, 4) if precision is not None else None,
                "recall": round(recall, 4),
                "ci_precision": [round(x, 4) if not np.isnan(x) else None for x in ci_p],
                "ci_recall": [round(x, 4) if not np.isnan(x) else None for x in ci_r],
                "excluded_from_headline": bool(excluded),
                "exclusion_reason": reason,
                "small_n_warning": n_boot_note,
            })
    return table


our_table = build_table(scored_injected, natural_flags, system="our")
base_table = build_table(scored_injected, natural_flags, system="base")

headline_cells = [t for t in our_table if not t["excluded_from_headline"]]
logger.info(f"headline (non-excluded) cells for our method: {len(headline_cells)} / {len(our_table)}")

## Results

Print the per-cell table and plot recall by category for OUR method vs. BASELINE, averaged across languages. On this small curated demo subset, expect the `negation_polarity_flip` category to show the clearest OUR-method-vs-baseline gap (the baseline has no negation resource at all, so its recall there is exactly 0.0) -- the same category the full run reports as the strongest headline result.

In [ ]:
print(f"{'language':8} {'category':28} {'sys':5} {'n':>3} {'recall':>7} {'precision':>9} {'excluded':>9}")
print("-" * 78)
for t_our, t_base in zip(our_table, base_table):
    for t in (t_our, t_base):
        r = "n/a" if t["recall"] is None else f"{t['recall']:.3f}"
        p = "n/a" if t["precision"] is None else f"{t['precision']:.3f}"
        print(f"{t['language']:8} {t['category']:28} {t['system']:5} {t['n_injected']:>3} {r:>7} {p:>9} {str(t['excluded_from_headline']):>9}")

n_headline_our = sum(1 for t in our_table if not t["excluded_from_headline"])
n_headline_base = sum(1 for t in base_table if not t["excluded_from_headline"])
print(f"\nheadline (non-excluded) cells: our={n_headline_our}/{len(our_table)}  baseline={n_headline_base}/{len(base_table)}")

# --- recall by category, averaged across languages, our vs. baseline ---
cats = cm.CATEGORY_KEYS
our_recall_by_cat = {c: np.mean([t["recall"] for t in our_table if t["category"] == c and t["recall"] is not None]) for c in cats}
base_recall_by_cat = {c: np.mean([t["recall"] for t in base_table if t["category"] == c and t["recall"] is not None]) for c in cats}

x = np.arange(len(cats))
width = 0.35
fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(x - width / 2, [our_recall_by_cat[c] for c in cats], width, label="our method")
ax.bar(x + width / 2, [base_recall_by_cat[c] for c in cats], width, label="baseline")
ax.axhline(USABLE_THRESHOLD, color="gray", linestyle="--", linewidth=1, label=f"usable threshold ({USABLE_THRESHOLD})")
ax.set_xticks(x)
ax.set_xticklabels(cats, rotation=20, ha="right")
ax.set_ylabel("recall (mean over languages)")
ax.set_title("Recall by category: our method vs. language-naive baseline (demo subset)")
ax.set_ylim(0, 1.05)
ax.legend()
fig.tight_layout()
plt.show()

logger.info(f"total demo runtime: {time.time() - t_start:.1f}s")
del scored_injected, natural_flags
gc.collect()